# Full Dataset EDA

This notebook profiles the full local challenge datasets, including the complete transaction history. It is separate from the 914-row platform submission file.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / 'Datasets'
DOCS_DIR = PROJECT_ROOT / 'Docs'

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

print(PROJECT_ROOT)

d:\projects\Data-Storm-2026


## Load Raw Data

In [2]:
outlets = pd.read_csv(DATASETS_DIR / 'outlet_master.csv')
coords = pd.read_csv(DATASETS_DIR / 'outlet_coordinates.csv')
transactions = pd.read_csv(DATASETS_DIR / 'transactions_history_final.csv')
seasonality = pd.read_csv(DATASETS_DIR / 'distributor_seasonality_details.csv')
holidays = pd.read_csv(DATASETS_DIR / 'holiday_list.csv')

datasets = {
    'outlet_master': outlets,
    'outlet_coordinates': coords,
    'transactions_history': transactions,
    'distributor_seasonality': seasonality,
    'holiday_list': holidays,
}

overview = pd.DataFrame([
    {
        'dataset': name,
        'rows': len(df),
        'columns': len(df.columns),
        'duplicate_rows': int(df.duplicated().sum()),
        'memory_mb': round(df.memory_usage(deep=True).sum() / (1024 * 1024), 2),
    }
    for name, df in datasets.items()
])
overview

,dataset,rows,columns,duplicate_rows,memory_mb
0,outlet_master,20000,4,0,3.34
1,outlet_coordinates,20000,3,0,1.41
2,transactions_history,2376389,7,0,460.51
3,distributor_seasonality,360,4,0,0.05
4,holiday_list,349,3,93,0.07


## Data Quality Profile

In [3]:
missing = []
for name, df in datasets.items():
    for column, count in df.isna().sum().items():
        if count:
            missing.append({
                'dataset': name,
                'column': column,
                'missing_rows': int(count),
                'missing_pct': round(100 * count / len(df), 3),
            })
missing = pd.DataFrame(missing)
missing

,dataset,column,missing_rows,missing_pct
0,outlet_master,Outlet_Size,196,0.98


In [4]:
outlet_type_map = {'Grocry': 'Grocery', 'Bakry': 'Bakery', 'Eatery ': 'Eatery'}
outlet_size_map = {'small': 'Small', '': 'Unknown'}

outlets_eda = outlets.copy()
outlets_eda['Outlet_Size_Normalized'] = (
    outlets_eda['Outlet_Size'].astype('string').fillna('').str.strip().replace(outlet_size_map).replace('', 'Unknown')
)
outlets_eda['Outlet_Type_Normalized'] = (
    outlets_eda['Outlet_Type'].astype('string').fillna('').str.strip().replace(outlet_type_map)
)

print('Raw outlet sizes')
display(outlets['Outlet_Size'].fillna('<missing>').astype(str).str.strip().value_counts(dropna=False))
print('Raw outlet types')
display(outlets['Outlet_Type'].fillna('<missing>').astype(str).str.strip().value_counts(dropna=False))

Raw outlet sizes


Outlet_Size
Small          9672
Medium         5702
Large          2887
Extra Large     943
small           600
<missing>       196
Name: count, dtype: int64

Raw outlet types


Outlet_Type
Eatery      2867
Hotel       2797
Grocery     2768
SMMT        2723
Kiosk       2691
Pharmacy    2691
Bakery      2678
Bakry        395
Grocry       390
Name: count, dtype: int64

In [5]:
coords_eda = coords.copy()
coords_eda['Latitude'] = pd.to_numeric(coords_eda['Latitude'], errors='coerce')
coords_eda['Longitude'] = pd.to_numeric(coords_eda['Longitude'], errors='coerce')
coords_eda['valid_coordinates'] = coords_eda['Latitude'].between(5.5, 10.2) & coords_eda['Longitude'].between(79.0, 82.1)

coordinate_quality = pd.DataFrame([
    {'metric': 'rows', 'value': len(coords_eda)},
    {'metric': 'unique_outlets', 'value': coords_eda['Outlet_ID'].nunique()},
    {'metric': 'duplicate_outlet_ids', 'value': int(coords_eda.duplicated('Outlet_ID').sum())},
    {'metric': 'valid_coordinate_rows', 'value': int(coords_eda['valid_coordinates'].sum())},
    {'metric': 'invalid_coordinate_rows', 'value': int((~coords_eda['valid_coordinates']).sum())},
    {'metric': 'min_latitude', 'value': coords_eda['Latitude'].min()},
    {'metric': 'max_latitude', 'value': coords_eda['Latitude'].max()},
    {'metric': 'min_longitude', 'value': coords_eda['Longitude'].min()},
    {'metric': 'max_longitude', 'value': coords_eda['Longitude'].max()},
])
coordinate_quality

,metric,value
0,rows,20000.000000
1,unique_outlets,20000.000000
2,duplicate_outlet_ids,0.000000
3,valid_coordinate_rows,19760.000000
4,invalid_coordinate_rows,240.000000
5,min_latitude,0.000000
6,max_latitude,80.792317
7,min_longitude,0.000000
8,max_longitude,80.799952


## Transaction EDA

In [6]:
tx = transactions.copy()
for column in ['Year', 'Month', 'Volume_Liters', 'Total_Bill_Value']:
    tx[column] = pd.to_numeric(tx[column], errors='coerce')

tx['valid_positive_transaction'] = (tx['Volume_Liters'] > 0) & (tx['Total_Bill_Value'] > 0)
tx_valid = tx.loc[tx['valid_positive_transaction']].copy()
tx_valid['value_per_liter'] = tx_valid['Total_Bill_Value'] / tx_valid['Volume_Liters']

transaction_quality = pd.DataFrame([
    {'metric': 'transaction_rows', 'value': len(tx)},
    {'metric': 'unique_outlets', 'value': tx['Outlet_ID'].nunique()},
    {'metric': 'unique_distributors', 'value': tx['Distributor_ID'].nunique()},
    {'metric': 'unique_skus', 'value': tx['SKU_ID'].nunique()},
    {'metric': 'non_positive_volume_rows', 'value': int((tx['Volume_Liters'] <= 0).sum())},
    {'metric': 'non_positive_bill_rows', 'value': int((tx['Total_Bill_Value'] <= 0).sum())},
    {'metric': 'rows_failing_positive_check', 'value': int((~tx['valid_positive_transaction']).sum())},
    {'metric': 'valid_rows', 'value': len(tx_valid)},
])
transaction_quality

,metric,value
0,transaction_rows,2376389
1,unique_outlets,20000
2,unique_distributors,10
3,unique_skus,10
4,non_positive_volume_rows,4853
5,non_positive_bill_rows,4753
6,rows_failing_positive_check,4853
7,valid_rows,2371536


In [7]:
monthly = tx_valid.groupby(['Outlet_ID', 'Year', 'Month'], as_index=False).agg(
    Monthly_Liters=('Volume_Liters', 'sum'),
    Monthly_Bill_Value=('Total_Bill_Value', 'sum'),
    SKU_Count=('SKU_ID', 'nunique'),
    Transaction_Lines=('SKU_ID', 'size'),
    Distributor_ID=('Distributor_ID', lambda values: values.mode().iat[0]),
)
monthly['Value_Per_Liter'] = monthly['Monthly_Bill_Value'] / monthly['Monthly_Liters']

print('Transaction volume distribution')
display(tx_valid['Volume_Liters'].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))
print('Outlet-month volume distribution')
display(monthly['Monthly_Liters'].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))

Transaction volume distribution


count    2.371536e+06
mean     5.283578e+01
std      9.537776e+01
min      1.237186e+00
1%       2.832088e+00
5%       4.217130e+00
10%      5.497375e+00
25%      1.024062e+01
50%      2.321772e+01
75%      5.453202e+01
90%      1.170959e+02
95%      1.974111e+02
99%      5.543613e+02
max      9.438578e+03
Name: Volume_Liters, dtype: float64

Outlet-month volume distribution


count    450588.000000
mean        278.085424
std         384.251539
min           1.263895
1%            4.222984
5%            9.954511
10%          16.205227
25%          36.977812
50%         102.217304
75%         354.492277
90%         750.503030
95%        1265.456456
99%        1846.728887
max       10457.941328
Name: Monthly_Liters, dtype: float64

In [8]:
monthly_totals = tx_valid.groupby(['Year', 'Month'], as_index=False).agg(
    transaction_rows=('Outlet_ID', 'size'),
    active_outlets=('Outlet_ID', 'nunique'),
    total_liters=('Volume_Liters', 'sum'),
    total_bill_value=('Total_Bill_Value', 'sum'),
)
monthly_totals['avg_liters_per_active_outlet'] = monthly_totals['total_liters'] / monthly_totals['active_outlets']
monthly_totals.round(3)

,Year,Month,transaction_rows,active_outlets,total_liters,total_bill_value,avg_liters_per_active_outlet
0,2023,1,65374,12498,3445811.058,9.043463e+08,275.709
1,2023,2,65530,12491,3469527.246,9.079770e+08,277.762
2,2023,3,66708,12492,3337347.615,8.763063e+08,267.159
3,2023,4,65825,12536,4763914.031,1.249379e+09,380.019
4,2023,5,65859,12483,2594021.444,6.814161e+08,207.804
5,2023,6,66036,12477,2623146.150,6.854267e+08,210.239
6,2023,7,65779,12508,3275888.205,8.609366e+08,261.903
7,2023,8,65872,12561,3601155.881,9.399039e+08,286.693
8,2023,9,66183,12638,3286698.142,8.653467e+08,260.065
9,2023,10,66172,12575,3305286.180,8.650808e+08,262.846


In [9]:
distributor_perf = tx_valid.groupby('Distributor_ID', as_index=False).agg(
    transaction_rows=('Outlet_ID', 'size'),
    active_outlets=('Outlet_ID', 'nunique'),
    total_liters=('Volume_Liters', 'sum'),
    avg_transaction_liters=('Volume_Liters', 'mean'),
).sort_values('total_liters', ascending=False)

sku_perf = tx_valid.groupby('SKU_ID', as_index=False).agg(
    transaction_rows=('Outlet_ID', 'size'),
    active_outlets=('Outlet_ID', 'nunique'),
    total_liters=('Volume_Liters', 'sum'),
    avg_value_per_liter=('value_per_liter', 'mean'),
).sort_values('total_liters', ascending=False)

display(distributor_perf.round(3))
display(sku_perf.round(3))

,Distributor_ID,transaction_rows,active_outlets,total_liters,avg_transaction_liters
7,DIST_W_01,365580,3020,1.891804e+07,51.748
9,DIST_W_03,362269,2991,1.879348e+07,51.877
8,DIST_W_02,364466,2989,1.835145e+07,50.352
3,DIST_NW_01,239332,2015,1.319030e+07,55.113
4,DIST_NW_02,235607,1985,1.266243e+07,53.744
0,DIST_C_01,163954,1385,9.118310e+06,55.615
2,DIST_C_03,152832,1296,8.909558e+06,58.296
6,DIST_S_02,168457,1495,8.758726e+06,51.994
5,DIST_S_01,167007,1505,8.367820e+06,50.105
1,DIST_C_02,152032,1319,8.231849e+06,54.146


,SKU_ID,transaction_rows,active_outlets,total_liters,avg_value_per_liter
5,SKU_06,231577,19963,5.725818e+07,84.000
1,SKU_02,231507,19953,1.716130e+07,253.329
6,SKU_07,232157,19927,1.146329e+07,649.997
4,SKU_05,231402,19950,1.144757e+07,89.999
0,SKU_01,287097,20000,6.498561e+06,339.480
7,SKU_08,231310,19931,5.723718e+06,439.997
3,SKU_04,231781,19949,4.585054e+06,325.000
2,SKU_03,231552,19947,4.580335e+06,349.996
9,SKU_10,231849,19949,3.722443e+06,369.226
8,SKU_09,231304,19945,2.861513e+06,2199.983


## Outlet-Level Business Cuts

In [10]:
outlet_sales = monthly.groupby('Outlet_ID').agg(
    active_months=('Monthly_Liters', 'size'),
    mean_monthly_liters=('Monthly_Liters', 'mean'),
    median_monthly_liters=('Monthly_Liters', 'median'),
    max_monthly_liters=('Monthly_Liters', 'max'),
    total_liters=('Monthly_Liters', 'sum'),
    mean_sku_count=('SKU_Count', 'mean'),
    mean_value_per_liter=('Value_Per_Liter', 'mean'),
).reset_index()

outlet_profile = (
    outlets_eda
    .merge(coords_eda[['Outlet_ID', 'valid_coordinates']], on='Outlet_ID', how='left')
    .merge(outlet_sales, on='Outlet_ID', how='left')
)

by_size = outlet_profile.groupby('Outlet_Size_Normalized').agg(
    outlets=('Outlet_ID', 'count'),
    avg_coolers=('Cooler_Count', 'mean'),
    zero_cooler_pct=('Cooler_Count', lambda s: 100 * (s == 0).mean()),
    avg_max_monthly_liters=('max_monthly_liters', 'mean'),
    median_max_monthly_liters=('max_monthly_liters', 'median'),
).round(3)

by_type = outlet_profile.groupby('Outlet_Type_Normalized').agg(
    outlets=('Outlet_ID', 'count'),
    avg_coolers=('Cooler_Count', 'mean'),
    avg_max_monthly_liters=('max_monthly_liters', 'mean'),
    median_max_monthly_liters=('max_monthly_liters', 'median'),
).round(3).sort_values('avg_max_monthly_liters', ascending=False)

display(by_size)
display(by_type)

,outlets,avg_coolers,zero_cooler_pct,avg_max_monthly_liters,median_max_monthly_liters
Outlet_Size_Normalized,,,,,
Extra Large,943,3.494,0.000,2039.854,2029.965
Large,2887,3.464,0.000,957.250,950.619
Medium,5702,1.496,0.000,320.774,308.365
Small,10272,0.363,67.884,134.112,115.073
Unknown,196,1.347,34.184,139.667,115.198


,outlets,avg_coolers,avg_max_monthly_liters,median_max_monthly_liters
Outlet_Type_Normalized,,,,
Pharmacy,2691,1.288,407.609,161.880
Kiosk,2691,1.289,401.940,170.057
Grocery,3158,1.274,400.875,164.878
Hotel,2797,1.307,399.928,163.619
Eatery,2867,1.297,394.810,166.817
Bakery,3073,1.295,389.474,160.714
SMMT,2723,1.288,378.022,161.878


## Calendar and Seasonality

In [11]:
holiday_profile = holidays.copy()
holiday_profile['Date'] = pd.to_datetime(holiday_profile['Date'], errors='coerce')
holiday_summary = pd.DataFrame([
    {'metric': 'holiday_rows', 'value': len(holiday_profile)},
    {'metric': 'exact_duplicate_rows', 'value': int(holiday_profile.duplicated().sum())},
    {'metric': 'duplicate_date_name_type_rows', 'value': int(holiday_profile.duplicated(['Date', 'Holiday_Name', 'Holiday_Type']).sum())},
    {'metric': 'unique_dates', 'value': holiday_profile['Date'].nunique()},
])

display(seasonality['Seasonality_Index'].value_counts())
display(holiday_summary)
display(holiday_profile['Holiday_Type'].value_counts())

Seasonality_Index
Moderate        249
Favorable        81
Un-Favorable     30
Name: count, dtype: int64

,metric,value
0,holiday_rows,349
1,exact_duplicate_rows,93
2,duplicate_date_name_type_rows,93
3,unique_dates,76


Holiday_Type
Bank          99
Public        98
Mercantile    93
Poya Day      59
Name: count, dtype: int64

## Main EDA Conclusions

- The raw data has 20,000 outlets and 2,376,389 transaction rows across 36 months.
- Outlet master contains clear legacy artifacts: missing sizes, lowercase `small`, `Grocry`, and `Bakry`.
- 240 coordinate rows fall outside plausible Sri Lankan bounds and should be quarantined before geospatial work.
- 4,853 transaction rows fail the positive volume or positive bill-value check.
- The wide gap between median and upper-percentile outlet monthly maxima supports a peer-frontier approach for latent potential.
- Holiday rows contain duplicates, so calendar features should be aggregated carefully.

A written summary of this EDA is available in `Docs/eda_summary.md`.